# PRiSM Track B Closed-Loop (Thin Colab Notebook)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/thesis-research-colab/blob/main/PRiSM_trackB_closedloop_simulation_colab.ipynb)

This notebook is intentionally thin. Core logic lives in `src/trackb/*.py`.


## Run Contract

- Pull latest code from GitHub at runtime start.
- Keep this notebook orchestration-only (no large inline modules).
- Persist outputs/checkpoints under Google Drive using `cfg.run_prefix`.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/achyutmorang/thesis-research-colab.git"
REPO_DIR = "/content/thesis-research-colab"

if os.path.exists(REPO_DIR):
    print("Repo exists, fast-forwarding main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# Deterministic setup (run ONCE in a fresh Colab runtime)
# Set RUN_SETUP=True for first run after Runtime -> Restart runtime.
RUN_SETUP = False

if RUN_SETUP:
    import os
    import subprocess
    import sys
    from pathlib import Path

    def sh(cmd: str) -> None:
        print(f"[setup] $ {cmd}")
        rc = subprocess.run(["bash", "-lc", cmd], check=False)
        if rc.returncode != 0:
            raise RuntimeError(f"Command failed ({rc.returncode}): {cmd}")

    # 1) Core package manager + Waymax (pins JAX ecosystem line)
    sh("pip install --upgrade pip")
    sh("pip install --upgrade 'git+https://github.com/waymo-research/waymax.git@main#egg=waymo-waymax'")

    # 2) GPU JAX in Waymax-compatible line (jax 0.7.x, numpy 2.x)
    sh("pip install --upgrade 'jax[cuda12]>=0.7.0,<0.8' -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")

    # 3) LatentDriver dependencies (transformers>=4.46 avoids numpy<2 pin)
    sh("pip install --upgrade mediapy seaborn scikit-learn tqdm")
    sh("pip install --upgrade hydra-core==1.3.2 omegaconf==2.3.0 lightning==2.3.3 pytorch-lightning==2.3.3 einops==0.8.0 transformers==4.46.3 huggingface_hub")

    # 4) LatentDriver source (avoid editable install with native extension build)
    sh("git clone --depth 1 https://github.com/Sephirex-X/LatentDriver.git /content/LatentDriver || true")

    if '/content/LatentDriver' not in sys.path:
        sys.path.insert(0, '/content/LatentDriver')
    os.environ['PYTHONPATH'] = '/content/LatentDriver:' + os.environ.get('PYTHONPATH', '')

    # 5) Patch transformers imports in LatentDriver for compatibility
    gpt2_model_path = Path('/content/LatentDriver/src/policy/latentdriver/gpt2_model.py')
    if gpt2_model_path.exists():
        gpt2_txt = gpt2_model_path.read_text()
        old_block = '''from transformers.modeling_utils import (
    Conv1D,
    PreTrainedModel,
    SequenceSummary,
    find_pruneable_heads_and_indices,
    prune_conv1d_layer,
)'''
        new_block = '''from transformers.modeling_utils import PreTrainedModel

try:
    from transformers.modeling_utils import SequenceSummary
except Exception:
    class SequenceSummary(nn.Module):
        def __init__(self, config=None):
            super().__init__()
        def forward(self, hidden_states, *args, **kwargs):
            return hidden_states[:, -1]

try:
    from transformers.modeling_utils import Conv1D, find_pruneable_heads_and_indices, prune_conv1d_layer
except Exception:
    from transformers.pytorch_utils import Conv1D, find_pruneable_heads_and_indices, prune_conv1d_layer
'''
        if old_block in gpt2_txt:
            gpt2_model_path.write_text(gpt2_txt.replace(old_block, new_block))
            print('[patch] gpt2_model transformers compatibility patch applied')
        else:
            print('[patch] gpt2_model block already patched or changed')

    # 6) Patch sort_vertices op fallback when CUDA extension is unavailable
    sort_vert_path = Path('/content/LatentDriver/src/ops/sort_vertices/sort_vert.py')
    if sort_vert_path.exists():
        fallback_code = '''import torch
from torch.autograd import Function

try:
    from src.ops.sort_vertices import sort_vertices as _sort_vertices_ext
except Exception:
    _sort_vertices_ext = None


def _sort_vertices_fallback(vertices, mask, num_valid):
    # vertices: [B,N,M,2], mask: [B,N,M], num_valid: [B,N]
    B, N, M, _ = vertices.shape
    out = torch.zeros((B, N, 9), dtype=torch.long, device=vertices.device)
    for b in range(B):
        for n in range(N):
            nv = int(num_valid[b, n].item())
            if nv <= 0:
                continue
            valid_idx = torch.nonzero(mask[b, n], as_tuple=False).flatten()
            if valid_idx.numel() == 0:
                continue
            valid_idx = valid_idx[:nv]
            pts = vertices[b, n, valid_idx]
            ang = torch.atan2(pts[:, 1], pts[:, 0])
            order = valid_idx[torch.argsort(ang)]
            k = min(int(order.numel()), 8)
            out[b, n, :k] = order[:k]
            out[b, n, k] = order[0]
            if k + 1 < 9:
                out[b, n, k + 1:] = order[k - 1]
    return out


class SortVertices(Function):
    @staticmethod
    def forward(ctx, vertices, mask, num_valid):
        if _sort_vertices_ext is not None:
            idx = _sort_vertices_ext.sort_vertices_forward(vertices, mask, num_valid)
        else:
            idx = _sort_vertices_fallback(vertices, mask, num_valid)
        ctx.mark_non_differentiable(idx)
        return idx

    @staticmethod
    def backward(ctx, gradout):
        return ()


sort_v = SortVertices.apply
'''
        sort_vert_path.write_text(fallback_code)
        print('[patch] sort_vert fallback patch applied')

    # 7) Fetch expected checkpoint if missing
    ckpt_path = Path('/content/checkpoints/lantentdriver_t2_J3.ckpt')
    if not ckpt_path.exists():
        ckpt_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            from huggingface_hub import hf_hub_download
            downloaded = hf_hub_download(
                repo_id='Sephirex-x/LatentDriver',
                filename='checkpoints/lantentdriver_t2_J3.ckpt',
                local_dir='/content',
            )
            print('[ckpt] downloaded:', downloaded)
        except Exception as e:
            print('[ckpt] hf_hub_download failed:', e)
            print('[ckpt] trying direct URL fallback...')
            rc = subprocess.run([
                'bash', '-lc',
                "wget -O '/content/checkpoints/lantentdriver_t2_J3.ckpt' 'https://huggingface.co/Sephirex-x/LatentDriver/resolve/main/checkpoints/lantentdriver_t2_J3.ckpt'"
            ], check=False)
            if rc.returncode != 0:
                print('[ckpt] direct download failed; place checkpoint manually at', ckpt_path)

    if ckpt_path.exists():
        print('[ckpt] ready:', ckpt_path, 'size_mb=', round(ckpt_path.stat().st_size / (1024 ** 2), 2))
    else:
        print('[ckpt] missing:', ckpt_path)

    print('Setup complete. Restart runtime once, set RUN_SETUP=False, then Run all.')
else:
    print('RUN_SETUP=False: skipping dependency setup.')


In [ ]:
# Manual checkpoint fallback (run only if setup could not fetch checkpoint)
# !mkdir -p /content/checkpoints
# !wget -O /content/checkpoints/lantentdriver_t2_J3.ckpt https://huggingface.co/Sephirex-x/LatentDriver/resolve/main/checkpoints/lantentdriver_t2_J3.ckpt
# !ls -lh /content/checkpoints/lantentdriver_t2_J3.ckpt


In [ ]:
import numpy as np
import pandas as pd

from src.trackb import (
    SearchConfig,
    build_trackb_runner_and_splits,
    configure_persistent_run_prefix,
    export_trackb_artifacts,
    initialize_configs,
    make_waymax_data_iter,
    run_preflight_and_calibration,
    run_surprise_quality_gate,
    run_trackb_closed_loop,
    summarize_method_outputs,
)


In [ ]:
cfg, search_cfg, ckpt_scan_df = initialize_configs()

RUN_TAG = "trackb_closedloop_v1"
PERSIST_ROOT = "/content/drive/MyDrive/prism_trackb_runs"
N_SHARDS = 1
SHARD_ID = 0
RESTORE_FROM_UPLOAD = False

run_prefix = configure_persistent_run_prefix(
    cfg=cfg,
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    shard_id=SHARD_ID,
    n_shards=N_SHARDS,
)
print("run_prefix =", run_prefix)

if len(ckpt_scan_df):
    display(ckpt_scan_df.head(10))


In [ ]:
dataset_config, data_iter = make_waymax_data_iter(cfg)
(
    runner,
    data,
    train_idx,
    test_idx,
    eval_idx_all,
    eval_idx,
    reference_df,
    base_eval_openloop_df,
) = build_trackb_runner_and_splits(
    cfg=cfg,
    data_iter=data_iter,
    dataset_config=dataset_config,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
)

print(f"train={len(train_idx)} test={len(test_idx)} eval_shard={len(eval_idx)}")


In [ ]:
(
    preflight_df,
    closedloop_calib_df,
    trackb_thresholds,
    calib_diag_df,
    calib_quant_df,
) = run_preflight_and_calibration(
    runner=runner,
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    reference_df=reference_df,
    restore_from_upload=RESTORE_FROM_UPLOAD,
)

print("Calibration thresholds:", trackb_thresholds)
display(preflight_df)
display(calib_diag_df)
display(calib_quant_df)


In [ ]:
gate_summary_df, dist_change_summary_df = run_surprise_quality_gate(
    closedloop_calib_df=closedloop_calib_df,
    surprise_gate_enabled=True,
)
display(gate_summary_df)
display(dist_change_summary_df)


In [ ]:
trackb_results_df, trackb_trace_df = run_trackb_closed_loop(
    runner=runner,
    eval_idx=eval_idx,
    cfg=cfg,
    search_cfg=search_cfg,
    thresholds=trackb_thresholds,
    run_prefix=cfg.run_prefix,
    static_frames={
        "base_eval_openloop_df": base_eval_openloop_df,
        "reference_df": reference_df,
        "closedloop_calib_df": closedloop_calib_df,
        "preflight_df": preflight_df,
        "calib_diag_df": calib_diag_df,
        "calib_quant_df": calib_quant_df,
    },
)

print("Result rows:", len(trackb_results_df))
print("Trace rows:", len(trackb_trace_df))


In [ ]:
quick_summary_df, sanity_df, fairness_checks_df, trace_diag_df = summarize_method_outputs(
    trackb_results_df=trackb_results_df,
    trackb_trace_df=trackb_trace_df,
)

artifact_paths = export_trackb_artifacts(
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    trackb_results_df=trackb_results_df,
    trackb_trace_df=trackb_trace_df,
    base_eval_openloop_df=base_eval_openloop_df,
    reference_df=reference_df,
    closedloop_calib_df=closedloop_calib_df,
    preflight_df=preflight_df,
    calib_diag_df=calib_diag_df,
    calib_quant_df=calib_quant_df,
    trackb_thresholds=trackb_thresholds,
    quick_summary_df=quick_summary_df,
    sanity_df=sanity_df,
    fairness_checks_df=fairness_checks_df,
    trace_diag_df=trace_diag_df,
)

display(quick_summary_df)
display(sanity_df)
display(fairness_checks_df)
display(trace_diag_df)

print("Artifacts:")
for key, value in artifact_paths.items():
    print(f" - {key}: {value}")
